# Phase 1 Demo: Digital Biomarkers for Relapse Prediction

This notebook demonstrates the Digital Biomarkers Agent for passive monitoring and relapse prediction using LSTM models.

**Key Features:**
- Passive data collection from wearables (Fitbit, Apple Watch)
- LSTM-based relapse forecasting (AUC ~0.75)
- Sleep efficiency, HRV, and activity monitoring
- Real-time alert triggering

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from irip.agents.digital_biomarkers_agent import (
    DigitalBiomarkersAgent,
    generate_mock_fitbit_data
)

## Initialize Agent

In [ ]:
# Create and initialize agent
agent = DigitalBiomarkersAgent()
await agent.initialize()

print(f"Agent Status: {agent.state}")
print(f"Model Parameters: {sum(p.numel() for p in agent.model.parameters())}")

## Generate Mock Wearable Data

Simulates 7 days of Fitbit data with gradual decline (relapse prodrome pattern)

In [ ]:
# Generate mock data
readings = generate_mock_fitbit_data(days=7)

# Convert to DataFrame for visualization
data_dict = {
    'timestamp': [],
    'biomarker': [],
    'value': []
}

for reading in readings:
    data_dict['timestamp'].append(reading.timestamp)
    data_dict['biomarker'].append(reading.biomarker_type.value)
    data_dict['value'].append(reading.value)

df = pd.DataFrame(data_dict)
print(f"\nGenerated {len(readings)} biomarker readings over 7 days")
print(df.head(10))

## Visualize Biomarker Trends

In [ ]:
# Plot trends
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Sleep efficiency
sleep_data = df[df['biomarker'] == 'sleep_efficiency']
axes[0].plot(sleep_data['timestamp'], sleep_data['value'], marker='o', color='blue')
axes[0].axhline(y=0.70, color='r', linestyle='--', label='Alert Threshold')
axes[0].set_ylabel('Sleep Efficiency')
axes[0].set_title('Sleep Efficiency Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Activity level
activity_data = df[df['biomarker'] == 'activity_level']
axes[1].plot(activity_data['timestamp'], activity_data['value'], marker='s', color='green')
axes[1].set_ylabel('Activity Level')
axes[1].set_title('Physical Activity Over Time')
axes[1].grid(True, alpha=0.3)

# HRV
hrv_data = df[df['biomarker'] == 'heart_rate_variability']
axes[2].plot(hrv_data['timestamp'], hrv_data['value'], marker='^', color='orange')
axes[2].set_ylabel('HRV (ms)')
axes[2].set_xlabel('Date')
axes[2].set_title('Heart Rate Variability Over Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Predict Relapse Risk

In [ ]:
# Perform risk assessment
patient_id = readings[0].patient_id
assessment = await agent.assess_relapse_risk(patient_id, readings)

print("\n" + "="*60)
print("RELAPSE RISK ASSESSMENT")
print("="*60)
print(f"Patient ID: {assessment.patient_id}")
print(f"Timestamp: {assessment.timestamp}")
print(f"\nRisk Score: {assessment.risk_score:.3f}")
print(f"Risk Level: {assessment.risk_level.value.upper()}")
print(f"95% CI: [{assessment.confidence_interval_95[0]:.3f}, {assessment.confidence_interval_95[1]:.3f}]")
print(f"\nAlert Triggered: {'YES' if assessment.alert_triggered else 'NO'}")

print(f"\nContributing Factors:")
for factor, score in assessment.contributing_factors.items():
    print(f"  - {factor}: {score:.3f}")

print(f"\nRecommended Actions:")
for i, action in enumerate(assessment.recommended_actions, 1):
    print(f"  {i}. {action}")

## Time-Series Forecasting

In [ ]:
# Prepare time series data
timeseries_data = df.pivot_table(
    index='timestamp',
    columns='biomarker',
    values='value'
).resample('H').mean().interpolate()

# Forecast relapse
forecast = await agent.forecast_relapse(timeseries_data)

print("\n" + "="*60)
print("24-HOUR RELAPSE FORECAST")
print("="*60)
print(f"Risk Score: {forecast['risk']:.3f}")
print(f"95% Confidence Interval: {forecast['95_ci']}")
print(f"Alert: {'TRIGGERED' if forecast['alert'] else 'None'}")
if forecast['alert_reason']:
    print(f"Reason: {forecast['alert_reason']}")

## Visualize Risk Prediction

In [ ]:
# Create risk visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Plot risk score with confidence interval
risk = forecast['risk']
ci_lower, ci_upper = forecast['95_ci']

ax.barh(['Relapse Risk'], [risk], color='red' if risk > 0.5 else 'orange' if risk > 0.25 else 'green')
ax.errorbar([risk], ['Relapse Risk'], xerr=[[risk-ci_lower], [ci_upper-risk]], 
            fmt='o', color='black', capsize=5, capthick=2)

# Add threshold lines
ax.axvline(x=0.25, color='yellow', linestyle='--', alpha=0.5, label='Moderate Risk')
ax.axvline(x=0.50, color='orange', linestyle='--', alpha=0.5, label='High Risk')
ax.axvline(x=0.75, color='red', linestyle='--', alpha=0.5, label='Very High Risk')

ax.set_xlabel('Risk Score')
ax.set_xlim(0, 1.0)
ax.set_title('24-Hour Relapse Risk Prediction', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This demo shows:
1. ✅ Passive biomarker collection from wearables
2. ✅ LSTM-based relapse prediction with confidence intervals
3. ✅ Automatic alert triggering for high-risk patterns
4. ✅ Actionable clinical recommendations

**Clinical Integration:**
- Feeds into Master Orchestrator for treatment adjustments
- Triggers crisis intervention protocols when needed
- Enables proactive rather than reactive care